In [0]:

%pip install faker
dbutils.library.restartPython()

In [0]:
from pyspark.sql.functions import col, row_number
from pyspark.sql.window import Window

bronze_drivers = spark.table("lakehouse_db.bronze_drivers")

window_spec = Window.partitionBy("id").orderBy(col("ingested_at").desc())

silver_drivers_df = bronze_drivers \
    .withColumn("rn", row_number().over(window_spec)) \
    .filter(col("rn") == 1) \
    .drop("rn") \
    .filter(col("id").isNotNull())


silver_drivers_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("clusterBy", "experience") \
    .saveAsTable("lakehouse_db.silver_drivers")

print("Таблица lakehouse_db.silver_drivers успешно создана с применением Liquid Clustering!")

In [0]:
import random
from pyspark.sql.functions import udf, current_timestamp
from pyspark.sql.types import StringType, DoubleType, StructType, StructField
from faker import Faker

drivers_ids = [row.id for row in spark.table("lakehouse_db.silver_drivers").select("id").collect()]
vehicles_ids = [row.vehicle_id for row in spark.table("lakehouse_db.silver_vehicles").select("vehicle_id").collect()]


trip_schema = StructType([
    StructField("trip_id", StringType(), False),
    StructField("driver_id", StringType(), True),
    StructField("vehicle_id", StringType(), True),
    StructField("distance_km", DoubleType(), True),
    StructField("fare_amount", DoubleType(), True)
])


def generate_trip():
    fake_worker = Faker()
    dist = round(random.uniform(1.0, 45.0), 2)
    fare = round(dist * random.uniform(15.0, 25.0) + 50.0, 2)
    return {
        "trip_id": fake_worker.uuid4(),
        "driver_id": random.choice(drivers_ids) if drivers_ids else fake_worker.uuid4(),
        "vehicle_id": random.choice(vehicles_ids) if vehicles_ids else fake_worker.uuid4(),
        "distance_km": float(dist),
        "fare_amount": float(fare)
    }

generate_trip_udf = udf(generate_trip, trip_schema)


trips_df = spark.range(200) \
    .withColumn("t_data", generate_trip_udf()) \
    .select("t_data.*") \
    .withColumn("trip_timestamp", current_timestamp())


trips_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("lakehouse_db.silver_trips")

print("Таблица lakehouse_db.silver_trips успешно создана!")